In [1]:
import pandas as pd
from scipy import stats

#### 일원 분산 분석
##### 귀무가설 : 네 가지 비료의 효과는 동일하다.
##### 대립가설 : 비료의 효과에는 차이가 있다.(적어도 두 가지 비료의 효과에는 차이가 있다.)

In [3]:
df = pd.DataFrame({
    'A': [10.5, 11.3, 10.8, 9.6, 11.1, 10.2, 10.9, 11.4, 10.5, 10.3],
    'B': [11.9, 12.4, 12.1, 13.2, 12.5, 11.8, 12.2, 12.9, 12.4, 12.3],
    'C': [11.2, 11.7, 11.6, 10.9, 11.3, 11.1, 10.8, 11.5, 11.4, 11.0],
    'D': [9.8, 9.4, 9.1, 9.5, 9.6, 9.9, 9.2, 9.7, 9.3, 9.4]
})
df.head()

,A,B,C,D
0,10.5,11.9,11.2,9.8
1,11.3,12.4,11.7,9.4
2,10.8,12.1,11.6,9.1
3,9.6,13.2,10.9,9.5
4,11.1,12.5,11.3,9.6


In [17]:
def check_shapiro(df):
    
    statistic,p_value = stats.shapiro(df)
    if p_value > 0.05:
        print(df.name,": 정규성 만족(",p_value,")")
    else:
        print(df.name,": 정규성 불만족",p_value,")")

print("===정규성 검정===")
check_shapiro(df['A'])
check_shapiro(df['B'])
check_shapiro(df['C'])
check_shapiro(df['D'])

===정규성 검정===
A : 정규성 만족( 0.8400161543468654 )
B : 정규성 만족( 0.6308700692815115 )
C : 정규성 만족( 0.892367306190296 )
D : 정규성 만족( 0.9346854448707653 )


In [19]:
statistic,p_value = stats.levene(df['A'],df['B'],df['C'],df['D'])

print("===등분산성 검정===")
print("p_value :", p_value)

===등분산성 검정===
p_value : 0.14127835331346628


In [21]:
statistic,p_value = stats.f_oneway(df['A'],df['B'],df['C'],df['D'])

print("===일원 분산 분석===")
print("p_value :", p_value)

if p_value > 0.05:
    print("귀무가설 채택")
else:
    print("귀무가설 기각")

===일원 분산 분석===
p_value : 1.001838152252373e-16
귀무가설 기각


#### 일원 분산 분석(ols 활용)

In [22]:
df = pd.read_csv('fertilizer.csv')
df.head()

,비료,성장
0,A,10.5
1,A,11.3
2,A,10.8
3,A,9.6
4,A,11.1


In [24]:
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

In [27]:
model = ols('성장~C(비료)', df).fit()
anova = anova_lm(model)
anova

,df,sum_sq,mean_sq,F,PR(>F)
C(비료),3.0,43.21875,14.406250,89.126139,1.001838e-16
Residual,36.0,5.81900,0.161639,NaN,NaN


#### 이원 분산 분석
##### 나무(주 효과)
##### 귀무가설 : 모든 나무 종류의 성장률은 동일한다.
##### 대립가설 : 나무 종류의 성잘률에는 차이가 있다.
##### 비료(주 효과)
##### 귀무가설 : 모든 비료의 성장률 효과는 동일하다.
##### 대립가설 : 비료의 성장률 효과에는 차이가 있다.
##### 상호작용
##### 귀무가설 : 나무 종류와 비료 간의 상호작용은 성장률에 영향을 주지 않는다.
##### 대립가설 : 나무 종류와 비료 간에는 성장률에 영향을 주는 상호작용이 있다.

In [28]:
df = pd.read_csv('tree.csv')
df.head()

,나무,비료,성장률
0,A,1,54.967142
1,A,1,48.617357
2,A,1,56.476885
3,A,1,65.230299
4,A,1,47.658466


In [30]:
model = ols('성장률~C(나무)+C(비료)+C(나무):C(비료)', df).fit()
anova = anova_lm(model)
anova

,df,sum_sq,mean_sq,F,PR(>F)
C(나무),3.0,4783.353938,1594.451313,18.855528,6.600012e-10
C(비료),2.0,1127.924259,563.962129,6.669256,1.857612e-03
C(나무):C(비료),6.0,717.520672,119.586779,1.414199,2.157357e-01
Residual,108.0,9132.639448,84.561476,NaN,NaN


In [35]:
p_values = [anova.iloc[0,-1],anova.iloc[1,-1],anova.iloc[2,-1]]

for val in p_values:
    if val > 0.05:
        print("귀무가설 채택(영향x) :",val)
    else:
        print('귀무가설 기각(영향o) :',val)

귀무가설 기각(영향o) : 6.600012072741466e-10
귀무가설 기각(영향o) : 0.0018576116097874565
귀무가설 채택(영향x) : 0.2157356676781794
